## Normalizes the MHC data 

In [104]:
import pandas as pd

unnorm_data = pd.read_excel("../data/MHC Trip Summary.xlsx")
unnorm_data

,Date,City,Physical location,Site Types,Latitude,Longitude,Organizer,Services offered,Total Pharmacists,Total APPE Students,...,Medication related problems identified,Patients referred.1,Patients referred to PCP.1,Patients referred to Healthwise MC.1,Patients referred to other sites.1,Patients needing f/u.1,Patients who were seen at f/u.1,Patients not seen at f/u.1,New diagnosis noted at f/u.1,Interventions made at appointment.1
0,2024-08-02 00:00:00,Ottawa,Ottawa Schools,Special Event,41.028836,-84.046572,HealthWise,BP + BG,3,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-08-06 00:00:00,Lima,Mercy Thrift Store,Thrift store,40.743039,-84.111477,HealthWise,MHC,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-08-07 00:00:00,Lima,Lima Library,Library,40.740543,-84.115236,HealthWise,MHC,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-08-13 00:00:00,Kenton,Kenton Seton,Senior community,40.654257,-83.591779,HealthWise,NaN,3,2,...,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-08-14 00:00:00,Bluffton,Bluffton Library,Library,40.892385,-83.892901,HealthWise,MHC,3,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110,2025-05-07 00:00:00,Lima,Christian Corner,Food Bank,40.712793,-84.107995,HealthWise,MHC,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,2025-05-08 00:00:00,Bluffton,Bluffton Library,Library,40.892385,-83.892901,HealthWise,NaN,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
112,2025-05-09 00:00:00,Wapakoneta,Wapak UCC,Church,40.569259,-84.195249,HealthWise,MHC,2,0,...,5.0,4.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,5/13/2025-5/15/2025,Ada,Employee screenings,Special Event,40.767245,-83.830893,HealthWise,Bone density + skin scope,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [105]:

no_special = unnorm_data.loc[:, "Site Types"] != "Special Event"

sites = unnorm_data.loc[no_special,["Physical location", "Site Types", "City"]]

# String manipulation
sites.loc[:,"Site Types"] =  sites.loc[:,"Site Types"].str.lower()

# Drop repeated rows
sites = sites.drop_duplicates()
sites = sites.dropna()
sites.index = sites.reset_index().index
sites.index.name = "siteId"

# Rename columns
sites.rename(columns={"Physical location" : "siteName", "Site Types": "siteType", "City": "city"}, inplace=True)

sites

,siteName,siteType,city
siteId,,,
0,Mercy Thrift Store,thrift store,Lima
1,Lima Library,library,Lima
2,Kenton Seton,senior community,Kenton
3,Bluffton Library,library,Bluffton
4,Lima ODB,charitable meal,Lima
5,Alger Food Pantry,food pantry,Alger
6,Saint Marks UMC,church,Lima
7,Dunkirk Dinner,charitable meal,Dunkirk
8,HardinCrest,senior community,Kenton


## Events

In [106]:
no_special = unnorm_data.loc[:, "Site Types"] != "Special Event"

# Keep "Physical location" and "City" to get "siteId" from sites table
events = unnorm_data.loc[no_special,["Physical location", "City", "Date", "Total patients screened"]]
events = events.drop_duplicates()

# You have to reset index in order to get "siteId" included on the merge
sites2merge = sites.reset_index().loc[:,["siteName", "city", "siteId"]]

# Merge to get siteId 
events = pd.merge(events, sites2merge, right_on=["siteName", "city"], left_on=["Physical location", "City"])
# Drop unneeded columns 
events = events.drop(["siteName", "city"], axis=1)

# rename columns 
events.rename(columns={"Total patients screened": "footTraffic", "Date": "date"}, inplace=True)

events.index.name = "eventId"
events

,Physical location,City,date,footTraffic,siteId
eventId,,,,,
0,Mercy Thrift Store,Lima,2024-08-06 00:00:00,1,0
1,Lima Library,Lima,2024-08-07 00:00:00,3,1
2,Kenton Seton,Kenton,2024-08-13 00:00:00,3,2
3,Bluffton Library,Bluffton,2024-08-14 00:00:00,8,3
4,Lima ODB,Lima,2024-08-20 00:00:00,3,4
...,...,...,...,...,...
98,Christian Corner,Lima,2025-05-07 00:00:00,10,16
99,Bluffton Library,Bluffton,2025-05-08 00:00:00,7,3
100,Wapak UCC,Wapakoneta,2025-05-09 00:00:00,6,22


## Export

In [107]:
events.to_csv("../data/events.csv")
sites.to_csv("../data/sites.csv")
